# Groupby — split-apply-combine

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from _data import load_penguins

penguins = load_penguins()
penguins.shape

## 1. La logique split-apply-combine

Tout `groupby` suit toujours le même schéma en trois temps :

```
DataFrame original
┌──────────┬───────────┬────────────┐
│ species  │ island    │ body_mass_g │
├──────────┼───────────┼────────────┤
│ Adelie   │ Torgersen │ 3750       │
│ Adelie   │ Dream     │ 3800       │
│ Gentoo   │ Biscoe    │ 5000       │
│ Chinstrap│ Dream     │ 3500       │
│ Gentoo   │ Biscoe    │ 4800       │
└──────────┴───────────┴────────────┘
          │
     1. SPLIT — groupby("species")
          │
    ┌─────┴──────┐─────────────┐
    │ Adelie     │ Gentoo      │ Chinstrap
    │ [3750,3800]│ [5000,4800] │ [3500]
          │
     2. APPLY — .mean()
          │
    ┌─────┴──────┐─────────────┐
    │   3775     │   4900      │ 3500
          │
     3. COMBINE — résultat
          │
┌──────────┬────────────┐
│ species  │ body_mass_g │
├──────────┼────────────┤
│ Adelie   │ 3775       │
│ Chinstrap│ 3500       │
│ Gentoo   │ 4900       │
└──────────┴────────────┘
```

> L'objet `GroupBy` renvoyé par `.groupby()` ne fait **rien** — il décrit seulement le découpage.
> C'est l'appel à `.mean()`, `.sum()`, `.agg()` etc. qui déclenche le calcul.

## 2. Groupby simple

In [ ]:
# L'objet GroupBy — aucun calcul encore
g = penguins.groupby("species")
print(type(g))
print("Groupes :", list(g.groups.keys()))

In [ ]:
# Masse moyenne par espèce
penguins.groupby("species")["body_mass_g"].mean().round(0)

In [ ]:
# Clé composite : plusieurs colonnes
penguins.groupby(["species", "island"])["body_mass_g"].mean().round(0)

In [ ]:
# Aggregations classiques directement disponibles
penguins.groupby("species")["body_mass_g"].agg(["mean", "std", "min", "max", "count"]).round(1)

### Groupby et valeurs manquantes

Par défaut, `groupby` **exclut** les lignes dont la clé de groupe est `NaN`.
C'est un comportement silencieux qui peut fausser des comptages.

In [ ]:
print("NaN dans 'sex' :", penguins["sex"].isna().sum())

# Avec le comportement par défaut
n_defaut = penguins.groupby("sex")["body_mass_g"].count().sum()
print(f"Lignes comptées (dropna=True)  : {n_defaut}")

# En incluant les NaN dans les groupes
n_avec_nan = penguins.groupby("sex", dropna=False)["body_mass_g"].count().sum()
print(f"Lignes comptées (dropna=False) : {n_avec_nan}")

## 3. Aggregations multiples avec `.agg()`

`.agg()` accepte une liste de fonctions, un dictionnaire par colonne,
ou des fonctions personnalisées.

In [ ]:
# Dictionnaire : fonctions différentes par colonne
(
    penguins
    .groupby("species")
    .agg({
        "body_mass_g":    ["mean", "std"],
        "bill_length_mm": ["mean", "min", "max"],
        "flipper_length_mm": "mean",
    })
    .round(1)
)

In [ ]:
# Fonction personnalisée dans .agg()
def intervalle(s):
    """max - min"""
    return s.max() - s.min()

penguins.groupby("species")["body_mass_g"].agg(["mean", intervalle]).round(1)

## 4. Named aggregations — la syntaxe moderne

La syntaxe `col=("source", fonction)` nomme directement les colonnes du résultat.
C'est la forme recommandée depuis pandas 0.25 — lisible, chaînable, sans MultiIndex.

In [ ]:
(
    penguins
    .groupby("species")
    .agg(
        nb_individus       =("bill_length_mm", "count"),
        masse_moyenne_g    =("body_mass_g",    "mean"),
        masse_ecart_type   =("body_mass_g",    "std"),
        bec_longueur_moy   =("bill_length_mm", "mean"),
        nageoire_longueur  =("flipper_length_mm", "mean"),
    )
    .round(1)
)

In [ ]:
# Avantage : pas de MultiIndex dans les colonnes — immédiatement chaînable
result = (
    penguins
    .groupby(["species", "sex"])
    .agg(
        n          =("body_mass_g", "count"),
        masse_moy  =("body_mass_g", "mean"),
    )
    .round(0)
)
# Les colonnes sont 'n' et 'masse_moy', pas un MultiIndex (species, mean)
print(result.columns.tolist())
result

## 5. `groupby` + `assign` — chaîne moderne

Le résultat d'un `groupby().agg()` est une DataFrame normale.
On peut donc lui chaîner `.assign()`, `.query()`, `.sort_values()`…

In [ ]:
(
    penguins
    .groupby("species")
    .agg(
        n              =("body_mass_g", "count"),
        masse_moy      =("body_mass_g", "mean"),
        bec_moy        =("bill_length_mm", "mean"),
    )
    .assign(
        masse_moy   =lambda df_: df_["masse_moy"].round(0),
        bec_moy     =lambda df_: df_["bec_moy"].round(1),
        ratio_bec_masse=lambda df_: (df_["bec_moy"] / df_["masse_moy"] * 1000).round(2),
    )
    .sort_values("masse_moy", ascending=False)
    .reset_index()
)

## 6. `transform()` — garder la forme originale

`.agg()` réduit le DataFrame (une ligne par groupe).
`.transform()` retourne un résultat **de même longueur que le DataFrame original** —
chaque ligne reçoit la valeur agrégée de son groupe.

Cas d'usage typique : centrer une variable par rapport à la moyenne du groupe.

In [ ]:
# Masse de chaque individu, centrée par rapport à la moyenne de son espèce
penguins["masse_centree"] = (
    penguins["body_mass_g"]
    - penguins.groupby("species")["body_mass_g"].transform("mean")
)

penguins[["species", "body_mass_g", "masse_centree"]].head(8)

In [ ]:
# Vérification : la moyenne de masse_centree est 0 par groupe
penguins.groupby("species")["masse_centree"].mean().round(10)

In [ ]:
# Autre usage courant : rang au sein du groupe
penguins["rang_masse_dans_espece"] = (
    penguins.groupby("species")["body_mass_g"]
    .transform(lambda df_: df_.rank(ascending=False))
)

(
    penguins[["species", "body_mass_g", "rang_masse_dans_espece"]]
    .dropna()
    .sort_values(["species", "rang_masse_dans_espece"])
    .head(8)
)

## 7. `filter()` sur les groupes — garder ou rejeter des groupes entiers

`.filter()` appliqué à un GroupBy évalue une condition sur **chaque groupe**
et retourne uniquement les lignes des groupes qui passent la condition.
Le résultat a la même forme que le DataFrame original (toutes les colonnes),
mais avec potentiellement moins de lignes.

In [ ]:
# Garder uniquement les îles qui ont au moins 100 observations
penguins.groupby("island").filter(lambda df_: len(df_) >= 100).groupby("island").size()

In [ ]:
# Garder uniquement les espèces dont la masse moyenne dépasse 4 kg
(
    penguins
    .groupby("species")
    .filter(lambda df_: df_["body_mass_g"].mean() > 4000)
    .groupby("species")["body_mass_g"]
    .mean()
    .round(0)
)

## 8. `observed=True` pour les Categoricals

Quand la clé de groupe est un `Categorical`, `groupby` crée par défaut
un groupe pour **chaque catégorie définie**, même si elle n'a aucune observation.
Utilisez `observed=True` pour ne garder que les groupes réellement présents.

In [ ]:
penguins_cat = penguins.copy()
# On définit une catégorie avec une valeur qui n'existe pas dans les données
penguins_cat["species"] = pd.Categorical(
    penguins["species"],
    categories=["Adelie", "Chinstrap", "Gentoo", "Emperor"]  # Emperor n'existe pas
)

# observed=False (défaut) : groupe vide pour Emperor
print("observed=False :")
print(penguins_cat.groupby("species", observed=False)["body_mass_g"].count())

print("\nobserved=True :")
print(penguins_cat.groupby("species", observed=True)["body_mass_g"].count())

## Récapitulatif

| Besoin | Méthode |
|---|---|
| Une agrégation, une colonne | `.groupby("k")["col"].mean()` |
| Plusieurs agrégations nommées | `.groupby("k").agg(nom=("col", func), ...)` |
| Fonctions différentes par colonne | `.groupby("k").agg({"col1": [...], "col2": [...]})` |
| Valeur agrégée ramenée sur chaque ligne | `.groupby("k")["col"].transform("mean")` |
| Rang dans le groupe | `.groupby("k")["col"].transform(lambda df_: df_.rank())` |
| Garder/rejeter des groupes entiers | `.groupby("k").filter(lambda df_: condition)` |
| Inclure les NaN dans les clés | `.groupby("k", dropna=False)` |
| Catégoriels : seulement les groupes présents | `.groupby("k", observed=True)` |

> `apply()` sur groupby n'est pas présenté ici — il est couvert dans le notebook 5 (`apply / map / transform`),
> avec une discussion sur les cas où il est vraiment nécessaire.